# Бонус: Data Temporality — часове перерахування впевненості (temporal rescoring)
Сусідні кадри майже однакові (EDA §8: IoU одного об'єкта між кадрами t і t+1 = 0.90). Справжній об'єкт
детектується в сусідніх кадрах у тому самому місці, а хибна детекція «блимає».
Для кожної детекції шукаємо в сусідніх кадрах (±w) бокс того ж класу з IoU ≥ τ і змішуємо впевненості:

`s' = (1 − α) · s + α · mean_n max_j s_j^n`  (0, якщо в кадрі n пари немає)

Параметри (w, τ, α) підбираються на val; найкращі застосовуються до тесту. Лише постпроцесинг збережених предиктів,
без GPU. Input: дані змагання + Output ноутбуків з `val_preds.pkl` / `test_preds.pkl`. Accelerator: None.

## 1. Середовище, оцінка

In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("pycocotools") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycocotools"])

import contextlib, io, itertools, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

NAMES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
MAX_DET = 500
RUN = "A1_yolo11s_1280"                 # чиї предикти покращуємо (найкращий на public LB)
ROOT = next(p.parent.parent for p in Path("/kaggle/input").rglob("val/annotations_coco.json"))
PRED_DIR = next(p.parent for p in Path("/kaggle/input").rglob("val_preds.pkl") if p.parent.name == RUN)
OUT = Path("/kaggle/working"); test_df = pd.read_csv(ROOT / "test_images.csv")
print("дані:", ROOT, "| предикти:", PRED_DIR)

with contextlib.redirect_stdout(io.StringIO()):
    GT = COCO(str(ROOT / "val" / "annotations_coco.json"))
KEY2ID = {(Path(i["file_name"]).parent.name, Path(i["file_name"]).stem): i["id"] for i in GT.dataset["images"]}


def evaluate(preds):
    dets = []
    for k, (b, s, c) in preds.items():
        for i in np.argsort(-s)[:MAX_DET]:
            x1, y1, x2, y2 = map(float, b[i])
            dets.append(dict(image_id=KEY2ID[k], category_id=int(c[i]) + 1, bbox=[x1, y1, x2 - x1, y2 - y1], score=float(s[i])))
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(GT, GT.loadRes(dets), "bbox")
        E.params.imgIds = [KEY2ID[k] for k in preds]
        E.params.maxDets = [1, 100, MAX_DET]
        E.evaluate(); E.accumulate(); E.summarize()
    p = E.eval["precision"][:, :, :, 0, -1]
    return dict(mAP=float(p[p > -1].mean()), AP50=float(E.stats[1]), AP_small=float(E.stats[3]), AR500=float(E.stats[8]))


def load(tag):
    raw = pickle.load(open(PRED_DIR / f"{tag}_preds.pkl", "rb"))
    return {k: (np.asarray(b, np.float32), np.asarray(s, np.float32), np.asarray(c, np.int16)) for k, (b, s, c) in raw.items()}


val, test = load("val"), load("test")
print(f"val кадрів: {len(val)} | test кадрів: {len(test)}")

дані: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026 | предикти: /kaggle/input/notebooks/papajonas/notebook9eb4aef616/artifacts/A1_yolo11s_1280
val кадрів: 1425 | test кадрів: 3811


## 2. Temporal rescoring

In [2]:
def iou_np(a, b):
    ix = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
    iy = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
    inter = ix * iy
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)


def neighbour_support(preds, window, tau, topk=300):
    """Для кожної детекції: середній по сусідніх кадрах (±window) максимальний score пари того ж класу з IoU ≥ tau."""
    by_clip = {}
    for k in preds:
        by_clip.setdefault(k[0], []).append(k)
    support = {}
    for clip, keys in by_clip.items():
        keys = sorted(keys, key=lambda k: int(k[1]))
        for i, k in enumerate(keys):
            b, s, c = preds[k]
            nb = [keys[j] for j in range(max(0, i - window), min(len(keys), i + window + 1)) if j != i]
            acc = np.zeros(len(b), np.float32)
            for kn in nb:
                bn, sn, cn = preds[kn]
                o = np.argsort(-sn)[:topk]; bn, sn, cn = bn[o], sn[o], cn[o]
                if len(b) == 0 or len(bn) == 0:
                    continue
                m = iou_np(b, bn) * (c[:, None] == cn[None, :])
                acc += np.where(m >= tau, sn[None, :], 0).max(1)
            support[k] = acc / max(1, len(nb))
    return support


def rescore(preds, support, alpha):
    return {k: (b, (1 - alpha) * s + alpha * support[k], c) for k, (b, s, c) in preds.items()}

## 3. Підбір (w, τ, α) на val

In [3]:
base = evaluate(val)
rows = [dict(window=0, tau=np.nan, alpha=0.0, **base)]
for window, tau in itertools.product([1, 2], [0.3, 0.5]):
    t0 = time.time()
    sup = neighbour_support(val, window, tau)
    for alpha in [0.2, 0.4, 0.6]:
        rows.append(dict(window=window, tau=tau, alpha=alpha, **evaluate(rescore(val, sup, alpha))))
    print(f"w={window} τ={tau}: {time.time() - t0:.0f} с")
grid = pd.DataFrame(rows)
grid["ΔmAP"] = grid.mAP - base["mAP"]
display(grid.round(4).sort_values("mAP", ascending=False))
best = grid.iloc[grid.mAP.idxmax()]
print(f"\nбаза: mAP {base['mAP']:.4f} → найкраще: w={int(best.window)}, τ={best.tau}, α={best.alpha}: mAP {best.mAP:.4f} "
      f"({best['ΔmAP'] * 100:+.2f} п.п.)")

w=1 τ=0.3: 592 с
w=1 τ=0.5: 589 с
w=2 τ=0.3: 595 с
w=2 τ=0.5: 593 с

база: mAP 0.1908 → найкраще: w=0, τ=nan, α=0.0: mAP 0.1908 (+0.00 п.п.)


,window,tau,alpha,mAP,AP50,AP_small,AR500,ΔmAP
0,0,NaN,0.0,0.1908,0.3787,0.1110,0.3177,0.0000
1,1,0.3,0.2,0.1893,0.3721,0.1082,0.3177,-0.0015
7,2,0.3,0.2,0.1892,0.3724,0.1081,0.3177,-0.0016
4,1,0.5,0.2,0.1889,0.3728,0.1093,0.3177,-0.0019
10,2,0.5,0.2,0.1883,0.3733,0.1097,0.3177,-0.0026
2,1,0.3,0.4,0.1860,0.3621,0.1036,0.3177,-0.0049
8,2,0.3,0.4,0.1857,0.3626,0.1039,0.3177,-0.0051
5,1,0.5,0.4,0.1852,0.3631,0.1058,0.3177,-0.0056
11,2,0.5,0.4,0.1835,0.3630,0.1066,0.3177,-0.0074
3,1,0.3,0.6,0.1783,0.3418,0.0972,0.3177,-0.0125


## 4. Застосування найкращих параметрів до тесту → сабміт
Кроки між тестовими кадрами — 1 кадр (у val — 2), тож вікно w у кадрах тесту подвоюється, щоб покривати той самий проміжок часу.

In [4]:
def write_submission(preds, path):
    rows = []
    for iid, fn in zip(test_df.image_id, test_df.file_name):
        k = (Path(fn).parent.name, Path(fn).stem)
        b, s, c = preds.get(k, (np.zeros((0, 4)), np.zeros(0), np.zeros(0)))
        w, h = b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]
        keep = np.where((w >= 1) & (h >= 1))[0]
        o = keep[np.argsort(-s[keep])][:MAX_DET]
        rows.append((iid, " ".join(f"{int(c[i])} {s[i]:.4f} {b[i, 0]:.1f} {b[i, 1]:.1f} {w[i]:.1f} {h[i]:.1f}" for i in o)))
    pd.DataFrame(rows, columns=["image_id", "PredictionString"]).to_csv(path, index=False)
    print("сабміт:", path)


if best.alpha > 0:
    sup_t = neighbour_support(test, int(best.window) * 2, best.tau)
    write_submission(rescore(test, sup_t, best.alpha), OUT / f"submission_{RUN}_temporal.csv")
grid.to_csv(OUT / f"temporal_grid_{RUN}.csv", index=False)